# KAN-RNN Session-Based Recommendation on YOOCHOOSE


In [ ]:
print("Notebook scaffold created. See subsequent cells for full pipeline.")


## Problem and Method
We implement session-based next-item prediction with RNN, GRU, and custom KAN-RNN where h_t = KAN([x_t; h_{t-1}]).


In [ ]:
import os, random, warnings, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from dataclasses import dataclass
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
@dataclass
class Config:
    data_path='./data/yoochoose-clicks.dat'
    subset_path='./data/yoochoose-clicks-1_64.dat'
    min_session_len=2
    min_item_support=5
    max_session_len=20
    batch_size=256
    emb_dim=100
    hidden_dim=100
    lr=1e-3
    weight_decay=1e-6
    epochs=10
    patience=3
    grad_clip=5.0
    topk=20
    grid_size=8
    spline_order=2
    use_linear_residual=True
    output_dir='./outputs_kan_rnn_yoochoose'
cfg=Config(); os.makedirs(cfg.output_dir, exist_ok=True)


In [ ]:
cols=['session_id','timestamp','item_id','category']
def load_data(cfg):
    p=Path(cfg.data_path)
    if not p.exists(): p=Path(cfg.subset_path)
    if not p.exists(): raise FileNotFoundError('Not available: dataset missing in ./data')
    df=pd.read_csv(p, header=None, names=cols)
    df['timestamp']=pd.to_datetime(df['timestamp'])
    return df
try:
    df_raw=load_data(cfg); print(len(df_raw)); display(df_raw.head())
except Exception as e:
    print('Not available', e); df_raw=None


In [ ]:
if df_raw is not None:
    sl=df_raw.groupby('session_id').size(); isup=df_raw.groupby('item_id').size()
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].hist(sl.clip(upper=50),bins=50); ax[0].set_title('Session lengths')
    ax[1].hist(isup.clip(upper=200),bins=50); ax[1].set_title('Item support')
    plt.show()
else: print('Not available')


In [ ]:
def preprocess(df,cfg):
    df=df.sort_values(['session_id','timestamp']).copy()
    s=df.groupby('session_id').size(); df=df[df.session_id.isin(s[s>=cfg.min_session_len].index)]
    f=df.groupby('item_id').size(); df=df[df.item_id.isin(f[f>=cfg.min_item_support].index)]
    s=df.groupby('session_id').size(); df=df[df.session_id.isin(s[s>=cfg.min_session_len].index)]
    item2idx={it:i+1 for i,it in enumerate(df.item_id.unique())}; df['item_idx']=df.item_id.map(item2idx)
    sessions=df.groupby('session_id')['item_idx'].apply(list).tolist()
    sessions=[x[-cfg.max_session_len:] for x in sessions if len(x)>=2]
    n=len(sessions); nv=int(0.1*n); nt=int(0.1*n)
    train, val, test = sessions[:n-nv-nt], sessions[n-nv-nt:n-nt], sessions[n-nt:]
    return df,item2idx,train,val,test

def pairs(sessions):
    X,y=[],[]
    for s in sessions:
        for t in range(1,len(s)): X.append(s[:t]); y.append(s[t])
    return X,y
if df_raw is not None:
    df_p,item2idx,tr,va,te=preprocess(df_raw,cfg)
    Xtr,ytr=pairs(tr); Xva,yva=pairs(va); Xte,yte=pairs(te)
    n_items=len(item2idx)+1
    print(len(Xtr),len(Xva),len(Xte),n_items)
else: print('Not available')


In [ ]:
class SessionDataset(Dataset):
    def __init__(self,X,y): self.X,self.y=X,y
    def __len__(self): return len(self.X)
    def __getitem__(self,i): return self.X[i],self.y[i]

def collate_fn(batch):
    seqs,targets=zip(*batch)
    lens=torch.tensor([len(s) for s in seqs],dtype=torch.long)
    T=lens.max().item(); x=torch.zeros(len(seqs),T,dtype=torch.long)
    for i,s in enumerate(seqs): x[i,:len(s)]=torch.tensor(s)
    return x,lens,torch.tensor(targets,dtype=torch.long)
if df_raw is not None:
    train_loader=DataLoader(SessionDataset(Xtr,ytr),batch_size=cfg.batch_size,shuffle=True,collate_fn=collate_fn)
    val_loader=DataLoader(SessionDataset(Xva,yva),batch_size=cfg.batch_size,shuffle=False,collate_fn=collate_fn)
    test_loader=DataLoader(SessionDataset(Xte,yte),batch_size=cfg.batch_size,shuffle=False,collate_fn=collate_fn)


In [ ]:
class BSplineKANLayer(nn.Module):
    def __init__(self,in_dim,out_dim,grid_size=8,spline_order=2,use_linear=True):
        super().__init__(); self.order=spline_order; self.use_linear=use_linear
        self.n_basis=grid_size+spline_order
        self.coeff=nn.Parameter(torch.randn(out_dim,in_dim,self.n_basis)*0.01)
        self.base=nn.Linear(in_dim,out_dim) if use_linear else None
        self.register_buffer('knots', torch.linspace(0,1,grid_size+2*spline_order+1))
    def _basis(self,x):
        x=x.unsqueeze(-1); K=self.knots; p=self.order
        N=((x>=K[:-1])&(x<K[1:])).float(); N[...,-1]=(x[...,0]==K[-1]).float()
        for d in range(1,p+1):
            L=[]
            for i in range(len(K)-d-1):
                d1=K[i+d]-K[i]; d2=K[i+d+1]-K[i+1]
                t1=0 if d1==0 else ((x[...,0]-K[i])/d1)*N[...,i]
                t2=0 if d2==0 else ((K[i+d+1]-x[...,0])/d2)*N[...,i+1]
                L.append(t1+t2)
            N=torch.stack(L,dim=-1)
        return N
    def forward(self,x):
        b=self._basis(torch.sigmoid(x))
        y=torch.einsum('bin,oin->bo',b,self.coeff)
        if self.base is not None: y=y+self.base(x)
        return y

class KANRNNCell(nn.Module):
    def __init__(self,input_dim,hidden_dim,grid_size=8,spline_order=2,use_linear=True):
        super().__init__(); self.hidden_dim=hidden_dim
        self.kan=BSplineKANLayer(input_dim+hidden_dim,hidden_dim,grid_size,spline_order,use_linear)
    def forward(self,x_t,h_prev):
        return torch.tanh(self.kan(torch.cat([x_t,h_prev],dim=-1)))


In [ ]:
class KANRNNRec(nn.Module):
    def __init__(self,n_items,emb_dim,hidden_dim,grid_size=8,spline_order=2,use_linear=True):
        super().__init__(); self.emb=nn.Embedding(n_items,emb_dim,padding_idx=0)
        self.cell=KANRNNCell(emb_dim,hidden_dim,grid_size,spline_order,use_linear)
        self.out=nn.Linear(hidden_dim,n_items)
    def forward(self,x,lens):
        e=self.emb(x); B,T=e.shape[:2]; h=torch.zeros(B,self.cell.hidden_dim,device=x.device)
        for t in range(T):
            hn=self.cell(e[:,t,:],h); m=(t<lens).float().unsqueeze(-1); h=hn*m+h*(1-m)
        return self.out(h)

class RNNBaseline(nn.Module):
    def __init__(self,n_items,emb_dim,hidden_dim):
        super().__init__(); self.emb=nn.Embedding(n_items,emb_dim,padding_idx=0); self.rnn=nn.RNN(emb_dim,hidden_dim,batch_first=True); self.out=nn.Linear(hidden_dim,n_items)
    def forward(self,x,lens):
        p=nn.utils.rnn.pack_padded_sequence(self.emb(x),lens.cpu(),batch_first=True,enforce_sorted=False); _,h=self.rnn(p); return self.out(h[-1])

class GRUBaseline(nn.Module):
    def __init__(self,n_items,emb_dim,hidden_dim):
        super().__init__(); self.emb=nn.Embedding(n_items,emb_dim,padding_idx=0); self.gru=nn.GRU(emb_dim,hidden_dim,batch_first=True); self.out=nn.Linear(hidden_dim,n_items)
    def forward(self,x,lens):
        p=nn.utils.rnn.pack_padded_sequence(self.emb(x),lens.cpu(),batch_first=True,enforce_sorted=False); _,h=self.gru(p); return self.out(h[-1])


In [ ]:
def recall_mrr_at_k(logits,y,k=20):
    top=torch.topk(logits,k=k,dim=1).indices; y=y.view(-1,1); hits=(top==y)
    recall=hits.any(dim=1).float().mean().item()
    rr=torch.zeros(len(y),device=logits.device)
    for i in range(k): rr+=hits[:,i].float()/(i+1)
    return recall, rr.mean().item()

def run_epoch(model,loader,opt=None):
    tr=opt is not None; model.train() if tr else model.eval(); ce=nn.CrossEntropyLoss()
    L=R=M=N=0
    for x,l,y in loader:
        x,l,y=x.to(device),l.to(device),y.to(device); logits=model(x,l); loss=ce(logits,y)
        if tr:
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); opt.step()
        rec,mrr=recall_mrr_at_k(logits,y,cfg.topk); b=x.size(0)
        L+=loss.item()*b; R+=rec*b; M+=mrr*b; N+=b
    return L/N,R/N,M/N


In [ ]:
def train_model(model,name):
    opt=torch.optim.Adam(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay)
    best=-1; wait=0; hist=[]; path=os.path.join(cfg.output_dir,f'{name}_best.pt')
    for ep in range(1,cfg.epochs+1):
        tl,tr,tm=run_epoch(model,train_loader,opt); vl,vr,vm=run_epoch(model,val_loader,None)
        hist.append({'epoch':ep,'train_loss':tl,'val_recall@20':vr,'val_mrr@20':vm})
        print(name,ep,tl,vr,vm)
        if vr>best: best=vr; wait=0; torch.save(model.state_dict(),path)
        else:
            wait+=1
            if wait>=cfg.patience: break
    return pd.DataFrame(hist), path


In [ ]:
results={}; test_metrics={}
if df_raw is not None and len(Xtr)>0:
    models={
        'rnn':RNNBaseline(n_items,cfg.emb_dim,cfg.hidden_dim).to(device),
        'gru':GRUBaseline(n_items,cfg.emb_dim,cfg.hidden_dim).to(device),
        'kanrnn':KANRNNRec(n_items,cfg.emb_dim,cfg.hidden_dim,cfg.grid_size,cfg.spline_order,True).to(device)
    }
    paths={}
    for n,m in models.items():
        h,p=train_model(m,n); results[n]=h; paths[n]=p
    def eval_saved(model,path):
        model.load_state_dict(torch.load(path,map_location=device)); l,r,m=run_epoch(model,test_loader,None); return {'test_loss':l,'Recall@20':r,'MRR@20':m,'HitRate@20':r}
    for n,m in models.items(): test_metrics[n]=eval_saved(m,paths[n])
    pd.DataFrame(test_metrics).T.to_csv(os.path.join(cfg.output_dir,'test_metrics.csv'))
    display(pd.DataFrame(test_metrics).T)
else: print('Not available')


In [ ]:
ablation=[]
if df_raw is not None and len(Xtr)>0:
    settings=[('KAN_no_spline',8,2,True,True),('KAN_grid4_order2',4,2,True,False),('KAN_grid8_order2',8,2,True,False),('KAN_grid8_order3',8,3,True,False)]
    for name,gs,so,ul,nos in settings:
        m=KANRNNRec(n_items,cfg.emb_dim,cfg.hidden_dim,gs,so,ul).to(device)
        if nos:
            with torch.no_grad(): m.cell.kan.coeff.zero_()
        h,p=train_model(m,name)
        m.load_state_dict(torch.load(p,map_location=device)); l,r,mm=run_epoch(m,test_loader,None)
        ablation.append({'setting':name,'Recall@20':r,'MRR@20':mm})
    ablation_df=pd.DataFrame(ablation); ablation_df.to_csv(os.path.join(cfg.output_dir,'ablation_metrics.csv'),index=False); display(ablation_df)
else: print('Not available')


In [ ]:
if results:
    plt.figure(figsize=(9,4))
    for n,h in results.items(): plt.plot(h['epoch'],h['train_loss'],label=n)
    plt.legend(); plt.title('Training loss'); plt.savefig(os.path.join(cfg.output_dir,'training_loss.png')); plt.show()
    plt.figure(figsize=(9,4))
    for n,h in results.items(): plt.plot(h['epoch'],h['val_recall@20'],label=n)
    plt.legend(); plt.title('Val Recall@20'); plt.savefig(os.path.join(cfg.output_dir,'val_recall20.png')); plt.show()
    plt.figure(figsize=(9,4))
    for n,h in results.items(): plt.plot(h['epoch'],h['val_mrr@20'],label=n)
    plt.legend(); plt.title('Val MRR@20'); plt.savefig(os.path.join(cfg.output_dir,'val_mrr20.png')); plt.show()
if 'ablation_df' in locals() and len(ablation_df)>0:
    ablation_df.set_index('setting')[['Recall@20','MRR@20']].plot(kind='bar',figsize=(9,4)); plt.tight_layout(); plt.savefig(os.path.join(cfg.output_dir,'ablation_bar.png')); plt.show()


## Conclusions
Implemented required pipeline, models, ablations, metrics, visualizations, and checkpoint saving. If dataset is missing: Not available.
